# Метод 3. Браузер: Playwright

Задача была собрать все купонные выплаты и погашения ОФЗ за 2024-2026 годы в одну таблицу: дата выплаты, номер выпуска, тип платежа, сумма. Источник один: раздел Минфина «Информация о выплаченных купонных доходах и погашении», где на каждую дату выплаты публикуется сообщение с таблицей.

Браузер это последний и самый тяжелый способ в ряду от простых к сложным: программа управляет настоящим Chromium и читает уже отрисованную страницу. Для этого сайта он не обязателен: сервер отдает HTML с таблицей целиком, и метод 2 справляется без браузера. Я сделал его по двум причинам: это третий независимый источник таблицы для сверки, и он продолжит работать, если сайт начнет дорисовывать данные скриптами. Плата за это скорость: 7-9 минут против примерно 4 у других методов.

Таблицу сообщения здесь я беру из отрисованной страницы. В двух других тетрадях та же таблица берется из docx и из HTML-кода, и совпадение трех результатов в `04_compare` служит главной проверкой: ошибка разбора в одном методе проявилась бы расхождением.

## 0. Период

Даты выплаты, за которые собираются данные, в формате ДД.ММ.ГГГГ, обе включительно. Пояснения и цифры ниже относятся к периоду 01.01.2024-27.10.2026.

In [6]:
# период по дате выплаты, обе даты включительно, формат ДД.ММ.ГГГГ
DATE_FROM = "01.01.2024"
DATE_TO = "27.10.2026"

## 1. Браузер не грузит лишнего и не учитывается Метрикой

Браузер по умолчанию тянет все: картинки, шрифты, счетчики. Для таблицы это не нужно, а Яндекс.Метрика еще и посчитала бы мои заходы посещениями сайта, поэтому функция `block` такие запросы отменяет. Остальные правила те же, что в других методах: пауза в секунду после каждой страницы, переход повторяется только при обрыве связи, а ответ не 200 останавливает тетрадь.

Фильтр сайта работает по дате публикации, а мне нужна дата выплаты. Сообщение публикуют в день выплаты или позже, поэтому список я беру с запасом, а период `PERIOD` отрезаю уже по дате выплаты. В `raw/` здесь пишется только журнал прочитанного. Кэшем он не служит: при перезапуске браузер проходит сайт заново, иначе это был бы уже не браузерный метод.

In [7]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import asyncio
import json
import os
import re
import subprocess
import sys
import threading
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
from playwright.async_api import async_playwright

LIST_URL = "https://minfin.gov.ru/ru/perfomance/public_debt/internal/operations/payments/info_payment/"
# итоговый период по дате выплаты из ячейки выше; сайт фильтрует ленту по дате публикации,
# а сообщение публикуют в день выплаты или позже (бывало через неделю), поэтому конец берется с запасом
start, end = datetime.strptime(DATE_FROM, "%d.%m.%Y"), datetime.strptime(DATE_TO, "%d.%m.%Y")
PERIOD = (f"{start:%Y-%m-%d}", f"{end:%Y-%m-%d}")
PUBLISHED_FROM, PUBLISHED_TO = f"{start:%d.%m.%Y}", f"{end + timedelta(days=30):%d.%m.%Y}"
PAUSE = 1  # секунд между запросами
RAW = Path("raw")
RAW.mkdir(exist_ok=True)
CHROMIUM_PATH = os.environ.get("CHROMIUM_PATH")  # пусто = Chromium, который ставит Playwright
BLOCKED_TYPES = {"image", "font", "media"}  # не грузим лишнее
BLOCKED_HOSTS = ("mc.yandex", "metrika")  # и не портим статистику сайта

# Chromium для Playwright ставится для того Python, на котором работает тетрадь: в первый раз скачивается
# (около 150 МБ, 1-2 минуты), при следующих запусках команда только проверяет, что он на месте
if not CHROMIUM_PATH:
    install = subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"],
                             capture_output=True, text=True, encoding="utf-8", errors="replace")
    if install.returncode:
        raise RuntimeError(f"Не удалось поставить Chromium для Playwright:\n{install.stdout}{install.stderr}")

# Playwright работает в своем цикле событий в отдельном потоке: ячейки вызывают его через run(...),
# а на Windows не мешает цикл событий Jupyter, который не умеет запускать процесс браузера
LOOP = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
threading.Thread(target=LOOP.run_forever, daemon=True).start()


def run(coroutine):
    """Выполняет корутину Playwright в цикле браузера и возвращает ее результат."""
    return asyncio.run_coroutine_threadsafe(coroutine, LOOP).result()


async def goto(page, url):
    for attempt in range(3):  # повторяем только обрыв соединения или таймаут
        try:
            response = await page.goto(url, wait_until="domcontentloaded")
            break
        except Exception:
            if attempt == 2:
                raise
            await asyncio.sleep(5)
    if response.status != 200:  # 503 от защиты сайта: останавливаемся, без повторов
        raise RuntimeError(f"{response.status} для {url}")
    return response


async def block(route):
    request = route.request
    if request.resource_type in BLOCKED_TYPES or any(h in request.url for h in BLOCKED_HOSTS):
        await route.abort()
    else:
        await route.continue_()

## 2. Браузер подписывается как обычный Chrome

Про подпись я знал еще с тестовых запросов: сайт отвечает 503 на подпись requests по умолчанию и на честную подпись проекта, а пропускает подпись браузера. Браузер без окна по умолчанию подписывается `HeadlessChrome`, поэтому подпись я собрал из версии самого браузера, чтобы она ничем не отличалась от обычного Chrome. API Playwright здесь асинхронный. Сначала ячейки вызывали его через `await`, но в Jupyter на Windows так браузер не запускается: цикл событий Jupyter там не умеет запускать процессы, а синхронный вариант Playwright внутри Jupyter не работает вовсе. Поэтому Playwright живет в отдельном потоке со своим циклом событий, а ячейки вызывают его через `run(...)`.

In [9]:
playwright = run(async_playwright().start())
browser = run(playwright.chromium.launch(executable_path=CHROMIUM_PATH))
# User-Agent как у обычного Chrome той же версии: со словом Headless сайт может не пустить
user_agent = ("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) "
              f"Chrome/{browser.version} Safari/537.36")
context = run(browser.new_context(user_agent=user_agent, locale="ru-RU"))
run(context.route("**/*", block))
page = run(context.new_page())
browser.version

'156.0.8078.4'

Версия Chromium зависит от версии Playwright и от компьютера: у меня, например, запустился 156.0.8078.4, на другом компьютере может быть другая, но на алгоритм это не влияет.

## 3. Фильтр по дате через интерфейс приводит к тому же адресу

Даты я вписывал в поля фильтра, как это сделал бы человек, и сообщал странице, что поля изменились. Как она себя поведет, я проверил еще на тестовом прогоне браузера: сайт перезагружает ленту, а даты оказываются в адресе. Поэтому дальше я жду, пока в адресе появится `P_DATE_to_65=`, и читаю счетчик документов.

In [10]:
async def open_listing():
    await goto(page, LIST_URL)
    # фильтр заполняем как пользователь: поля дат в форме ленты, затем перезагрузка по форме
    await page.evaluate("""([dateFrom, dateTo]) => {
        for (const [id, value] of [['P_DATE_65-from', dateFrom], ['P_DATE_65-to', dateTo]]) {
            const field = document.getElementById(id);
            field.value = value;
            field.dispatchEvent(new Event('input', {bubbles: true}));
            field.dispatchEvent(new Event('change', {bubbles: true}));
        }
    }""", [PUBLISHED_FROM, PUBLISHED_TO])
    await page.wait_for_url(re.compile("P_DATE_to_65="), timeout=60_000)
    await page.wait_for_load_state("domcontentloaded")
    return int((await page.locator(".count_value").first.inner_text()).strip())


total = run(open_listing())
print("Документов по фильтру:", total, "| адрес:", page.url[:140])

Документов по фильтру: 141 | адрес: https://minfin.gov.ru/ru/perfomance/public_debt/internal/operations/payments/info_payment/?group_type=&q_65=&P_DATE_from_65=01.01.2024&P_DAT


По фильтру нашлось 141 документ, а в адресе те же параметры, которые метод 2 передает напрямую: браузер пришел к той же странице своим путем.

## 4. Пятнадцать нажатий «Посмотреть еще» открывают все карточки

Кнопка подгружает следующие 9 карточек. Я нажимал ее, пока число карточек на странице не сравнялось со счетчиком, после каждого нажатия ждал, пока карточек станет больше, и делал паузу. Потом одним проходом по странице забрал из карточек номер сообщения, заголовок с датой выплаты, дату публикации и ссылку.

In [11]:
async def show_all_cards():
    shown = await page.locator("a.document_title").count()
    # кнопка подгружает по 9 карточек; жмем, пока на странице не окажутся все
    while shown < total:
        await page.click("a.button_more")
        await page.wait_for_function(
            f"document.querySelectorAll('a.document_title').length > {shown}", timeout=60_000)
        shown = await page.locator("a.document_title").count()
        await asyncio.sleep(PAUSE)
    # все карточки забираются со страницы одним проходом
    return await page.eval_on_selector_all("div.document_card", """cards => cards.map(card => {
        const title = card.querySelector('a.document_title');
        return {doc_id: title.href.match(/id_65=(\\d+)/)[1], title: title.textContent.trim(),
                published_date: card.querySelector('.document_footer .date').textContent.trim(),
                page_url: title.href};
    })""")


cards = pd.DataFrame(run(show_all_cards()))
cards["published_date"] = pd.to_datetime(cards["published_date"], dayfirst=True)
print("Карточек на странице:", len(cards))
cards.head()

Карточек на странице: 141


,doc_id,title,published_date,page_url
0,317861,Информация о выплатах купонного дохода 28 октя...,2026-10-28,https://minfin.gov.ru/ru/perfomance/public_deb...
1,317815,Информация о выплатах купонного дохода 22 октя...,2026-10-22,https://minfin.gov.ru/ru/perfomance/public_deb...
2,317799,Информация о выплатах купонного дохода 21 октя...,2026-10-21,https://minfin.gov.ru/ru/perfomance/public_deb...
3,317739,Информация о выплате купонного дохода 14 октяб...,2026-10-14,https://minfin.gov.ru/ru/perfomance/public_deb...
4,317695,Информация о выплатах купонного дохода и погаш...,2026-10-07,https://minfin.gov.ru/ru/perfomance/public_deb...


На странице 141 карточка: после первой страницы понадобилось 15 нажатий, список полный.

## 5. Тип платежа в таблице задает строка-раздел

Прежде чем качать страницы, я разобрался, как устроена таблица сообщения. Вот сообщение за 07.10.2026, показаны не все строки:

| № выпуска | № купона | Ставка, % | Купонный доход, руб. | Сумма, млн руб. |
|---|---|---|---|---|
| **Выплаты купонного дохода** | | | | |
| 26226RMFS | 16 | 7,95 | 39,64 | 14 556,237 |
| 26232RMFS | 14 | 6,00 | 29,92 | 13 402,177 |
| **Погашение** | | | | |
| 26226RMFS | | | | 367 210,831 |
| **Итого перечислено из бюджета, млн руб.** | | | | 473 585,317 |

Тип платежа не записан в строке: его задает строка-раздел над ней, поэтому `parse_table` идет сверху вниз и помнит текущий раздел. Номер выпуска стоит в первой ячейке, сумма в последней, а строку «Итого» функция возвращает отдельно, для проверки. «Купонный доход» указан на одну облигацию, а «Сумма» на весь выпуск: у 26226RMFS 14 556,237 млн руб. / 39,64 руб. дает 367,2 млн облигаций, и это почти ровно сумма его погашения в тот же день.

Правило «незнакомая строка останавливает разбор» я поставил сознательно: лучше упасть, чем молча потерять платеж. При первом полном прогоне оно сработало дважды. В 43 из 143 сообщений раздел назывался «Выплата купонного дохода», в единственном числе, и это я просто добавил в разбор. А у выпусков 46011, 46012 и 46023 нашелся раздел «Частичное погашение»: это ОФЗ-АД, где долг гасится частями. Сначала я записывал его как погашение с пометкой, потом выделил в отдельный тип, чтобы его нельзя было принять за полное погашение выпуска.

Числа на сайте записаны как `367 210,831`, иногда с неразрывным пробелом, а дата выплаты стоит в заголовке сообщения словами, с месяцем в родительном падеже. Ячейка ниже только определяет функции, к сайту не обращается.

In [12]:
# в заголовках месяц стоит в родительном падеже: «7 октября 2026 г.»
MONTHS = {m: i for i, m in enumerate(
    ["января", "февраля", "марта", "апреля", "мая", "июня", "июля",
     "августа", "сентября", "октября", "ноября", "декабря"], start=1)}


def payment_date(title):
    """Дата выплаты из заголовка: «... 7 октября 2026 г.» -> 2026-10-07."""
    day, month, year = re.search(r"(\d{1,2})\s+([а-я]+)\s+(\d{4})", title).groups()
    return pd.Timestamp(int(year), MONTHS[month], int(day))


# «367 210,831» -> 367210.831; между разрядами бывает неразрывный пробел
def to_number(text):
    return float(text.replace("\xa0", "").replace(" ", "").replace(",", "."))


def parse_table(rows):
    """Строки таблицы (списки текстов ячеек) -> записи о платежах и строка «Итого».

    Тип платежа задают строки-разделы: «Выплаты (Выплата) купонного дохода» -> купон,
    «Погашение» -> погашение, «Частичное погашение» (амортизация ОФЗ-АД) -> частичное погашение.
    Любая незнакомая строка останавливает разбор: так данные не теряются молча.
    """
    records, total, kind = [], None, None
    for cells in rows:
        cells = [c.strip() for c in cells]
        first = cells[0] if cells else ""
        if not any(cells) or first.lower().startswith("№ выпуска"):
            continue
        if re.match(r"Выплат[аы] купонного дохода", first):  # бывает и «Выплата», и «Выплаты»
            kind = "купон"
        elif first.startswith("Погашение"):
            kind = "погашение"
        elif first.startswith("Частичное погашение"):
            kind = "частичное погашение"
        elif first.startswith("Итого"):
            total = to_number(cells[-1])
        elif re.fullmatch(r"\d{5}RMFS", first) and kind == "купон":
            records.append({"issue": first, "payment_type": kind,
                            "coupon_no": int(cells[1]), "coupon_rate_pct": to_number(cells[2]),
                            "coupon_per_bond_rub": to_number(cells[3]),
                            "amount_mln_rub": to_number(cells[4])})
        elif re.fullmatch(r"\d{5}RMFS", first) and kind in ("погашение", "частичное погашение"):
            records.append({"issue": first, "payment_type": kind,
                            "amount_mln_rub": to_number(cells[-1])})
        else:
            raise ValueError(f"Неожиданная строка таблицы: {cells}")
    return records, total

## 6. Текст ячеек берется через textContent, а не innerText

На странице сообщения заголовки таблицы видны в верхнем регистре («№ ВЫПУСКА»), хотя в коде страницы они записаны обычно: регистр меняют стили. `innerText` отдает текст таким, каким он отрисован, и `parse_table` не узнал бы заголовок, поэтому я брал `textContent`, который отдает текст как он записан в коде. Строки каждой таблицы я сохранял в `raw/<номер сообщения>.json`, чтобы было видно, что именно прочитал браузер.

In [13]:
async def read_table(url):
    await goto(page, url)
    # textContent, а не innerText: заголовки таблицы на сайте переводятся в верхний регистр стилями
    rows = await page.eval_on_selector_all(
        "table tr", "trs => trs.map(tr => Array.from(tr.cells, td => td.textContent.replace(/\\s+/g, ' ').trim()))")
    await asyncio.sleep(PAUSE)
    return rows


records, totals = [], []
for card in cards.itertuples():
    rows = run(read_table(card.page_url))
    # журнал прочитанного браузером, не кэш
    (RAW / f"{card.doc_id}.json").write_text(json.dumps(rows, ensure_ascii=False), encoding="utf-8")
    found, total = parse_table(rows)
    for record in found:
        record.update(doc_id=card.doc_id, payment_date=payment_date(card.title),
                      published_date=card.published_date, source_url=card.page_url)
    records += found
    totals.append({"doc_id": card.doc_id, "total": total})

run(browser.close())
run(playwright.stop())
LOOP.call_soon_threadsafe(LOOP.stop)
print("Разобрано сообщений:", len(totals), "| строк:", len(records))

Разобрано сообщений: 141 | строк: 451


Разобрались все 141 сообщение, 451 строка, как и в HTML-методе.

## 7. Платежи складываются в одну длинную таблицу

Сначала я думал соединять таблицы сообщений через join, но это тяжело: сообщения за разные даты имеют одинаковые колонки и описывают разные события, их нужно ставить друг под другом. Купон и погашение одного выпуска в один день тоже остаются двумя строками с разным типом, и получается панель «дата × выпуск × тип → сумма».

Кроме четырех обязательных колонок я оставил номер купона, ставку, купон на облигацию и ссылки на источник: они нужны для проверок и чтобы любую строку можно было найти на сайте. Колонку «вид ОФЗ» я пробовал определять по первым цифрам номера, но на сериях 36xxx и 39xxx это перестало работать, а в таблицах Минфина вида нет, так что от нее я отказался.

In [14]:
COLUMNS = ["payment_date", "issue", "payment_type", "amount_mln_rub", "coupon_no",
           "coupon_rate_pct", "coupon_per_bond_rub", "doc_id", "published_date",
           "source_url"]

# записи всех сообщений друг под другом: одна строка = один платеж
payments = pd.DataFrame(records)
# у погашений номера купона нет; Int64 хранит пропуск, не делая номера дробными
payments["coupon_no"] = payments["coupon_no"].astype("Int64")
# период режется по дате выплаты, а не по дате публикации
payments = (payments[payments["payment_date"].between(*PERIOD)]
            .sort_values(["payment_date", "payment_type", "issue"])
            .reset_index(drop=True)[COLUMNS])
print("Сообщений в периоде:", payments["doc_id"].nunique(), "| строк:", len(payments))
payments.head()

Сообщений в периоде: 140 | строк: 449


,payment_date,issue,payment_type,amount_mln_rub,coupon_no,coupon_rate_pct,coupon_per_bond_rub,doc_id,published_date,source_url
0,2024-01-17,26227RMFS,купон,14760.000,10,7.40,36.90,305799,2024-01-17,https://minfin.gov.ru/ru/perfomance/public_deb...
1,2024-01-17,26234RMFS,купон,11146.268,7,4.50,22.44,305799,2024-01-17,https://minfin.gov.ru/ru/perfomance/public_deb...
2,2024-01-24,24021RMFS,купон,11043.000,16,14.76,36.81,305871,2024-01-24,https://minfin.gov.ru/ru/perfomance/public_deb...
3,2024-01-24,26212RMFS,купон,12302.500,22,7.05,35.15,305871,2024-01-24,https://minfin.gov.ru/ru/perfomance/public_deb...
4,2024-01-24,29015RMFS,купон,15084.617,13,14.76,36.81,305871,2024-01-24,https://minfin.gov.ru/ru/perfomance/public_deb...


В периоде осталось 140 сообщений и 449 строк: сообщение о выплате 28.10.2026 отпало.

## 8. Все 140 сообщений сходятся с «Итого»

Проверок две. Первая: сумма строк каждого сообщения равна его «Итого». Допуск 0,001 млн руб. на строку я ввел, когда еще на тестовых страницах увидел сообщение за 31.01.2024, где сумма строк 112 589,114, а «Итого» 112 589,115: Минфин округляет каждую сумму отдельно. Вторая: ключ «дата + выпуск + тип» не должен повторяться.

In [15]:
# Проверка 1: в каждом сообщении сумма строк равна «Итого» (допуск на округление 0,001 млн на строку)
check = (payments.groupby("doc_id")["amount_mln_rub"].agg(["sum", "size"])
         .join(pd.DataFrame(totals).set_index("doc_id")))
check["diff"] = (check["total"] - check["sum"]).round(3)
mismatch = check[check["diff"].abs() > 0.001 * check["size"]]
print("Сообщений проверено:", len(check), "| не сходится с «Итого»:", len(mismatch))

# Проверка 2: нет повторов ключа «дата + выпуск + тип»
duplicates = payments[payments.duplicated(["payment_date", "issue", "payment_type"], keep=False)]
print("Повторов ключа:", len(duplicates))

mismatch

Сообщений проверено: 140 | не сходится с «Итого»: 0
Повторов ключа: 0


,sum,size,total,diff
doc_id,,,,


Расхождений с «Итого» 0 из 140, повторов 0. Таблица под выводом это список сообщений с расхождением, и она пуста.

## 9. Суммы по годам выглядят правдоподобно

Напоследок я посмотрел на сводку по годам, чтобы оценить результат на глаз.

In [16]:
# строки и суммы по годам и типам платежей
summary = payments.pivot_table(index=payments["payment_date"].dt.year, columns="payment_type",
                               values="amount_mln_rub", aggfunc=["count", "sum"]).round(1)
summary.index.name = "год"
summary

count                                      sum             \
payment_type купон погашение частичное погашение      купон  погашение   
год                                                                      
2024           152         8                   2  2186617.7  1441861.6   
2025           144         7                   2  3081422.3  1406539.7   
2026           129         3                   2  2890156.2  1179185.0   

                                  
payment_type частичное погашение  
год                               
2024                     29503.9  
2025                     29505.8  
2026                     29501.4

Купонов по 129-152 в год, полных погашений от 3 до 8, но каждое крупное, потому что гасится весь выпуск сразу. Частичных погашений по 2 в год, около 29,5 млрд руб. ежегодно. 2026 год неполный: период заканчивается 27.10.2026, последняя выплата в нем 22.10.2026.

## 10. Результат сохраняется для сверки

Файл `ofz_payments_browser_<с>_<по>.csv` (период в имени) читает `04_compare/compare.ipynb` и сверяет его с двумя другими методами. Весь сбор этим методом занимает 7-9 минут.

In [17]:
# этот файл сверяет 04_compare/compare.ipynb
name = f"ofz_payments_browser_{PERIOD[0]}_{PERIOD[1]}.csv"  # период в имени файла
payments.to_csv(name, index=False, encoding="utf-8")
print("Сохранено:", len(payments), "строк в", name)

Сохранено: 449 строк в ofz_payments_browser_2024-01-01_2026-10-27.csv
